# Hawkes 2-Point Function — Full Pipeline Demo

**Model:** Nonlinear Hawkes process, 2 populations, delta-function synaptic filters, quadratic nonlinearity

**Observable:** 2-point correlation function $\langle \delta\dot{n}_1(t_1)\, \delta\dot{n}_2(t_2) \rangle$

**Loop levels:** Tree-level ($\ell = 0$) and 1-loop ($\ell = 1$)

---

## What this notebook does

This notebook walks through the full automated Feynman diagram pipeline (Phases A–G) for a concrete MSRJD field theory.

### Section 1 — Theory / Model Information
Loads the 2-population nonlinear Hawkes model and prints all of its defining data: response fields ($\tilde{n}_i$), physical fluctuation fields ($\delta\dot{n}_i$), parameters, synaptic kernels, operators, and the nonlinear transfer function $\phi$. Then expands the MSRJD action to the required Taylor order and extracts the free-action kernel matrix $\mathbf{K}$, its Fourier transform $\hat{\mathbf{K}}(\omega)$, the retarded propagator $\hat{\mathbf{G}}(\omega) = \hat{\mathbf{K}}^{-1}(\omega)$, its poles, residue matrices, and the time-domain propagator $\mathbf{G}(t)$.

### Section 2 — Prediagram Enumeration
For the 2-point function ($k=2$), enumerates all labeled trees, undirected topologies, and directed prediagrams at tree level ($\ell=0$) and 1-loop ($\ell=1$). Plots each topology and prediagram with color-coded vertices: black = external legs, red = source vertices (noise insertions, in-degree 0), light blue = internal interaction vertices.

### Section 3 — Vertex Extraction
Reads off all interaction vertex types (`VertexType`) and noise source types (`SourceType`) from the expanded action. Each vertex type has a bigrade $(n_{\tilde{}}, n_{\text{phys}})$, a list of response and physical legs, and a coefficient. Checks that the current Taylor order is sufficient to cover the maximum vertex degree appearing in the prediagrams.

### Section 4 — Prediagram Filtering
Discards any prediagram whose internal vertex degree signatures don't match any available vertex or source type. Reports how many prediagrams survive at each loop level and plots the survivors.

### Section 5 — Type Assignment (Fully Labeled Diagrams)
Sets up the external fields for the 2-point function ($\delta\dot{n}_1$, $\delta\dot{n}_2$) and builds the field-to-matrix-index maps. Then runs the constraint-satisfaction engine on each surviving prediagram: for every valid assignment of vertex types to internal vertices and field types to edges (consistent with the propagator matrix $\hat{\mathbf{G}}$), it produces a `TypedDiagram`. Each typed diagram is printed showing its external leg assignments, vertex types with coefficients, and edge propagator labels $\hat{G}_{ij}$.

### Section 6 — Unique Diagrams & Combinatorial Factor $M(\Gamma)$
Deduplicates typed diagrams to obtain the set of unique Feynman diagrams $\Gamma$. For each unique diagram, computes the combinatorial factor $M(\Gamma)$ — the number of distinct valid leg-to-edge attachments (bijections) that realize $\Gamma$. An *attachment* assigns each leg slot of each vertex to a specific incident edge; identical legs (same field type) can be permuted freely, giving $M(\Gamma) = \prod_v \prod_{\text{groups of } k \text{ identical legs}} k!$. The diagram's contribution to the $k$-point function is $M(\Gamma) \times \prod_v \text{coeff}(v) \times \int(\text{propagators})$. Reference: Helias & Dahmen, *Statistical Field Theory for Neural Networks*, Ch. 9.

In [ ]:
%display latex
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from IPython.display import display, Math
from sage.all import (
    SR, matrix, latex, I, pi, exp, diff, solve, integrate, oo,
    dirac_delta, function, var
)

from msrjd.core.field_theory import FieldTheory, fourier_transform, inverse_fourier_transform
from msrjd.core.vertices import extract_vertex_types, extract_source_types, available_degrees
from msrjd.core.cache import PipelineCache
from msrjd.enumeration.loop_diagram_enumeration import enumerate_all
from msrjd.diagrams.filter import filter_prediagrams, classify_prediagram_vertices
from msrjd.diagrams.type_assignment import (
    enumerate_typed_diagrams, enumerate_all as enumerate_all_typed,
    build_field_index_map, TypedDiagram
)
from msrjd.diagrams.causality import filter_causal
from msrjd.diagrams.symmetry import (
    combinatorial_factor, compute_all_combinatorial_factors,
    deduplicate_typed_diagrams,
)

from models.hawkes_sage import HAWKES_MODEL

print('Imports loaded.')

In [ ]:
# ── Pipeline cache ──────────────────────────────────────────────
# Set USE_CACHE = True to skip expensive re-computation on restart.
# Set USE_CACHE = False (or cache.clear()) to force a fresh run.
USE_CACHE = True

cache = PipelineCache('saved_results/hawkes_2pop_k2')
if USE_CACHE:
    print(f'Cache: {cache}')
    for e in cache.list_cached():
        print(f'  {e["key"]}  (saved {e["saved_at"][:19]})')
else:
    print('Cache disabled — all stages will be recomputed.')

---
## 1. Theory / Model Information

In [2]:
m = HAWKES_MODEL
print(f"Model:  {m['name']}")
print(f"Convention: {m['background_rate_convention']}")
print()

print('── Index sets ──')
for name, vals in m['index_sets'].items():
    print(f'  {name}: {list(vals)}')
print()

print('── Response fields (not expanded — full integration variables) ──')
for f in m['response_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Physical fields (expanded around MF background) ──')
for f in m['physical_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Parameters ──')
for p in m.get('parameters', []):
    idx = '(indexed)' if p.get('indexed') else '(scalar)'
    dom = f", domain={p['domain']}" if p.get('domain') else ''
    print(f"  {p['name']}  {idx}{dom}  — {p.get('description', '')}")
print()

print('── Kernels ──')
for k in m.get('kernels', []):
    print(f"  {k['name']}  — {k.get('description', '')}")
print()

print('── Operators ──')
for o in m.get('operators', []):
    print(f"  {o['name']}  (latex: ${o.get('latex_name', o['name'])}$)  — {o.get('description', '')}")
print()

print('── Nonlinear functions ──')
for f in m.get('functions', []):
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Specializations ──')
print('  φ quadratic (cubic and higher coefficients = 0)')
print('  g = δ(t)  (instantaneous synaptic coupling)')

Model:  Nonlinear Hawkes 2-population
Convention: ṅ_i* = -φ_i(v_i*)  [positive Poisson term:  +ñ_i ṅ_i + (e^{ñ_i}-1) φ_i]

── Index sets ──
  pop: [0, 1]

── Response fields (not expanded — full integration variables) ──
  nt  (latex: $\tilde{n}$)  — response field conjugate to spike train
  vt  (latex: $\tilde{v}$)  — response field conjugate to voltage

── Physical fields (expanded around MF background) ──
  dn  (latex: $\delta\dot{n}$)  — spike-train fluctuation around MF background
  dv  (latex: $\delta v$)  — voltage fluctuation around MF background

── Parameters ──
  nstar  (indexed), domain=positive  — background firing rate  nstar_i = phi_i(vstar_i)
  vstar  (indexed)  — background voltage
  E  (indexed)  — external drive
  tau  (scalar), domain=positive  — membrane time constant

── Kernels ──
  g  — synaptic filter kernel g(t)

── Operators ──
  Dt  (latex: $\partial_t$)  — ∂_t  (algebraic placeholder for the time-derivative operator)

── Nonlinear functions ──
  phi  (latex

### 1.1 Expand the action and show all sectors

In [3]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()

R  = ft.ring()
ns = ft._ns

print(f'Taylor order: {ft.taylor_order}')
print(f'Polynomial ring: {R}')
print(f'Ring generators: {R.gens()}')
print(f'Response generators (n_tilde={ft._n_tilde}): {R.gens()[:ft._n_tilde]}')
print(f'Physical generators: {R.gens()[ft._n_tilde:]}')
print()

passed = ft.sanity_check()
print()
ft.summary()

Taylor order: 4
Polynomial ring: Multivariate Polynomial Ring in nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2 over Symbolic Ring
Ring generators: (nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2)
Response generators (n_tilde=4): (nt1, nt2, vt1, vt2)
Physical generators: (dn1, dn2, dv1, dv2)

=== Sanity checks ===
  [PASS]  (n_tilde=0, n_phys=0)  constant term
  [PASS]  (n_tilde=1, n_phys=0)  tadpole — must vanish at MF saddle
  [PASS]  (n_tilde=0, n_phys=1)  linear physical-only — must vanish at EOM

=== Action sectors ===
  (n_tilde=1, n_phys=1)  [free action]:


<IPython.core.display.Math object>

  (n_tilde=1, n_phys=2)  [vertex (order 3)]:


<IPython.core.display.Math object>

  (n_tilde=2, n_phys=0)  [noise kernel]:


...

<IPython.core.display.Math object>

  (n_tilde=4, n_phys=1)  [vertex (order 5)]:


<IPython.core.display.Math object>

  (n_tilde=4, n_phys=2)  [vertex (order 6)]:


<IPython.core.display.Math object>

### 1.2 Propagator

Extract the kernel matrix $\mathbf{K}$ from the free action, Fourier transform, and invert.

In [4]:
import signal

S_free = ft.free_action()
ring_gen_names = [str(g) for g in R.gens()]

resp_names = [f'vt{i+1}' for i in ns.pop] + [f'nt{i+1}' for i in ns.pop]
phys_names = [f'dv{i+1}' for i in ns.pop] + [f'dn{i+1}' for i in ns.pop]

pos_to_row = {ring_gen_names.index(nm): i for i, nm in enumerate(resp_names)}
pos_to_col = {ring_gen_names.index(nm): j for j, nm in enumerate(phys_names)}

nf = len(resp_names)
K_data = [[SR(0)] * nf for _ in range(nf)]
for exp_vec, coeff in S_free.dict().items():
    row = col = None
    for k in range(len(ring_gen_names)):
        if exp_vec[k] > 0:
            if k in pos_to_row: row = pos_to_row[k]
            if k in pos_to_col: col = pos_to_col[k]
    if row is not None and col is not None:
        K_data[row][col] += SR(coeff)

K_mat = matrix(SR, K_data)

# Convert to kernel form
Dt       = ns.Dt
delta_D  = ns.delta_D
delta_Dp = ns.delta_Dp

def _to_kernel(c):
    c = SR(c)
    if c.has(delta_D) or c.has(ns.g):
        return c
    p0   = c.subs({Dt: 0})
    rest = (c - p0).subs({Dt: delta_Dp})
    return p0 * delta_D + rest

K_ker = matrix(SR, [[_to_kernel(K_mat[i, j]) for j in range(nf)] for i in range(nf)])

resp_sr  = [ns.vt[i] for i in ns.pop] + [ns.nt[i] for i in ns.pop]
phys_sr  = [ns.dv[i] for i in ns.pop] + [ns.dn[i] for i in ns.pop]

print('Field ordering:')
display(Math(r'\tilde{\mathbf{a}} = ' + latex(vector(resp_sr))
             + r',\qquad \mathbf{a} = ' + latex(vector(phys_sr))))
print()
print('Kernel matrix K (time domain):')
display(Math(r'\mathbf{K} = ' + latex(K_ker)))

# Fourier transform
t_var = SR.var('t')
omega = SR.var('omega', latex_name=r'\omega')

time_domain = {
    delta_D:  dirac_delta(t_var),
    delta_Dp: diff(dirac_delta(t_var), t_var),
    ns.g:     function('g')(t_var),
}

K_ft_data = [[SR(0)] * nf for _ in range(nf)]
for i in range(nf):
    for j in range(nf):
        c = K_ker[i, j]
        if not c.is_zero():
            K_ft_data[i][j] = fourier_transform(SR(c).subs(time_domain), t_var, omega)

K_ft = matrix(SR, K_ft_data)
print('Fourier-domain kernel:')
display(Math(r'\hat{\mathbf{K}}(\omega) = ' + latex(K_ft)))

# Propagator inverse
G_ft = K_ft.inverse().apply_map(lambda e: e.factor())
G_ft_explicit = True
print('Propagator:')
display(Math(r'\hat{\mathbf{G}}(\omega) = ' + latex(G_ft)))

# Adjugate, determinant, poles
adj_ft  = K_ft.adjugate()
D_omega = K_ft.det().expand()
D_prime = diff(D_omega, omega)

pole_eqs = solve(D_omega == 0, omega)
pole_vals = [eq.rhs() if hasattr(eq, 'rhs') else eq for eq in pole_eqs]

print(f'\ndet(K̂) = {D_omega}')
print(f'Poles ({len(pole_vals)}):')
for k, p in enumerate(pole_vals):
    display(Math(r'\omega_{' + str(k+1) + '} = ' + latex(p)))

# Residue matrices
C_mats = []
for omega_k in pole_vals:
    C_data = [[SR(0)] * nf for _ in range(nf)]
    for i in range(nf):
        for j in range(nf):
            n_ij = adj_ft[i, j]
            if not n_ij.is_zero():
                num = n_ij.subs({omega: omega_k})
                den = D_prime.subs({omega: omega_k})
                C_data[i][j] = (I * num / den).factor()
    C_mats.append(matrix(SR, C_data))

print('Residue matrices:')
for k, C in enumerate(C_mats):
    display(Math(r'\mathbf{C}_{' + str(k+1) + r'} = ' + latex(C)))

# Time-domain propagator
G_t = sum(C_mats[k] * exp(I * pole_vals[k] * t_var) for k in range(len(pole_vals)))
G_t = G_t.apply_map(lambda e: e.simplify_full())
print('Time-domain propagator G(t) for t > 0:')
display(Math(r'\mathbf{G}(t) = ' + latex(G_t)))

Field ordering:


<IPython.core.display.Math object>


Kernel matrix K (time domain):


<IPython.core.display.Math object>

Fourier-domain kernel:


...

Residue matrices:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Time-domain propagator G(t) for t > 0:


<IPython.core.display.Math object>

---
## 2. Prediagram Enumeration

Enumerate all trees, topologies, and prediagrams for the 2-point function at tree level ($\ell=0$) and 1-loop ($\ell=1$).

**Color code:** black = external legs, red = source vertices ($\mathrm{in}=0$), light blue = internal interaction vertices

In [5]:
from sage.plot.plot import graphics_array

def plot_prediagrams(pds, title_prefix=''):
    """Plot prediagrams with colored vertices."""
    if not pds:
        print('  (none)')
        return
    plots = []
    for i, (D, G, leaves, internal) in enumerate(pds):
        leaves_set = set(leaves)
        color_map = {}
        for v in D.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            elif D.in_degree(v) == 0:
                color_map.setdefault('red', []).append(v)
            else:
                color_map.setdefault('lightblue', []).append(v)
        plots.append(D.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

def plot_topologies(topos, title_prefix=''):
    """Plot undirected topologies."""
    if not topos:
        print('  (none)')
        return
    plots = []
    for i, (G, leaves, internal) in enumerate(topos):
        leaves_set = set(leaves)
        color_map = {}
        for v in G.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            else:
                color_map.setdefault('lightgray', []).append(v)
        plots.append(G.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

print('Plot helpers defined.')

Plot helpers defined.


### 2.1 Tree level ($\ell = 0$)

In [ ]:
k = 2

def _enumerate_tree():
    t, tp, pd, c = enumerate_all(k, 0, verbose=False)
    return (pd, c)

pds_0, counts_0 = cache.get_or_compute(
    'prediagrams', _enumerate_tree, k=k, loop_order=0
) if USE_CACHE else _enumerate_tree()

print(f'k={k}, ell=0:  {counts_0["n_trees"]} trees,  '
      f'{counts_0["n_topologies"]} topologies,  '
      f'{counts_0["n_prediagrams"]} prediagrams')

print('── Prediagrams ──')
plot_prediagrams(pds_0, title_prefix='PD ')

for i, (D, G, leaves, internal) in enumerate(pds_0):
    print(f'  PD {i+1}: edges={D.edges(labels=False)}')
    for v in sorted(D.vertices()):
        role = 'leaf' if v in leaves else ('source' if D.in_degree(v)==0 else 'internal')
        print(f'    v{v}: in={D.in_degree(v)}, out={D.out_degree(v)} ({role})')

### 2.2 One loop ($\ell = 1$)

In [ ]:
def _enumerate_1loop():
    t, tp, pd, c = enumerate_all(k, 1, verbose=False)
    return (pd, c)

pds_1, counts_1 = cache.get_or_compute(
    'prediagrams', _enumerate_1loop, k=k, loop_order=1
) if USE_CACHE else _enumerate_1loop()

print(f'k={k}, ell=1:  {counts_1["n_trees"]} trees,  '
      f'{counts_1["n_topologies"]} topologies,  '
      f'{counts_1["n_prediagrams"]} prediagrams')

print('── Prediagrams ──')
plot_prediagrams(pds_1, title_prefix='PD ')

for i, (D, G, leaves, internal) in enumerate(pds_1):
    print(f'  PD {i+1}: edges={D.edges(labels=False)}')
    for v in sorted(D.vertices()):
        role = 'leaf' if v in leaves else ('source' if D.in_degree(v)==0 else 'internal')
        print(f'    v{v}: in={D.in_degree(v)}, out={D.out_degree(v)} ({role})')

---
## 3. Vertex Extraction

Extract typed vertices (`VertexType`, `SourceType`) from the expanded action, based on the maximum vertex degree required by the prediagrams.

In [8]:
from msrjd.enumeration.degree_scan import max_vertex_degree, scan_source_vertices

all_pds = pds_0 + pds_1
max_deg = max_vertex_degree(all_pds)
src_degs = scan_source_vertices(all_pds)
print(f'Max vertex degree across all prediagrams: {max_deg}')
print(f'Source vertex out-degrees needed: {src_degs}')
print(f'Current Taylor order: {ft.taylor_order}')
print(f'Sufficient: {ft.taylor_order >= max_deg}')
print()

vtypes = extract_vertex_types(ft)
stypes = extract_source_types(ft)

print(f'── Interaction vertex types ({len(vtypes)}) ──')
for i, vt in enumerate(vtypes):
    print(f'  V{i+1}: bigrade={vt.bigrade}, in_deg={vt.in_degree}, out_deg={vt.out_degree}')
    print(f'        resp_legs={vt.response_legs}, phys_legs={vt.physical_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(vt.coefficient)}'))

print(f'\n── Source types ({len(stypes)}) ──')
for i, st in enumerate(stypes):
    print(f'  S{i+1}: bigrade={st.bigrade}, out_deg={st.out_degree}')
    print(f'        resp_legs={st.response_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(st.coefficient)}'))

int_degs, src_degs_avail = available_degrees(vtypes, stypes)
print(f'\nAvailable interaction degrees (in, out): {sorted(int_degs)}')
print(f'Available source out-degrees: {sorted(src_degs_avail)}')

Max vertex degree across all prediagrams: 4
Source vertex out-degrees needed: {2, 3}
Current Taylor order: 4
Sufficient: True

── Interaction vertex types (14) ──
  V1: bigrade=(4, 2), in_deg=2, out_deg=4
        resp_legs=[('nt', 1), ('nt', 1), ('nt', 1), ('nt', 1)], phys_legs=[('dv', 1), ('dv', 1)]


<IPython.core.display.Math object>

  V2: bigrade=(4, 2), in_deg=2, out_deg=4
        resp_legs=[('nt', 2), ('nt', 2), ('nt', 2), ('nt', 2)], phys_legs=[('dv', 2), ('dv', 2)]


<IPython.core.display.Math object>

  V3: bigrade=(4, 1), in_deg=1, out_deg=4
        resp_legs=[('nt', 1), ('nt', 1), ('nt', 1), ('nt', 1)], phys_legs=[('dv', 1)]


...

  S5: bigrade=(2, 0), out_deg=2
        resp_legs=[('nt', 1), ('nt', 1)]


<IPython.core.display.Math object>

  S6: bigrade=(2, 0), out_deg=2
        resp_legs=[('nt', 2), ('nt', 2)]


<IPython.core.display.Math object>


Available interaction degrees (in, out): [(1, 2), (1, 3), (1, 4), (2, 1), (2, 2), (2, 3), (2, 4)]
Available source out-degrees: [2, 3, 4]


---
## 4. Prediagram Filtering

Remove prediagrams whose vertex degree signatures don't match any available vertex or source type.

In [ ]:
def _filter_tree():
    kept, disc = filter_prediagrams(pds_0, vtypes, stypes)
    return (kept, disc)

def _filter_1loop():
    kept, disc = filter_prediagrams(pds_1, vtypes, stypes)
    return (kept, disc)

kept_0, disc_0 = cache.get_or_compute(
    'filtered', _filter_tree, k=k, loop_order=0
) if USE_CACHE else _filter_tree()

kept_1, disc_1 = cache.get_or_compute(
    'filtered', _filter_1loop, k=k, loop_order=1
) if USE_CACHE else _filter_1loop()

print('── Tree level (ell=0) ──')
print(f'  {len(pds_0)} prediagrams → {len(kept_0)} kept, {disc_0} discarded')

print()
print('── 1-loop (ell=1) ──')
print(f'  {len(pds_1)} prediagrams → {len(kept_1)} kept, {disc_1} discarded')

print()
print('── Surviving prediagrams (tree level) ──')
plot_prediagrams(kept_0, title_prefix='Tree PD ')

print('── Surviving prediagrams (1-loop) ──')
plot_prediagrams(kept_1, title_prefix='1L PD ')

---
## 5. Type Assignment — Fully Labeled Diagrams

Enumerate all valid field-type assignments on each surviving prediagram.

**External legs:** $\delta\dot{n}_1$ and $\delta\dot{n}_2$ (the 2-point function fields).  
**Edge convention:** each directed edge $u \to v$ carries propagator $\hat{G}_{ij}(\omega)$ where $i$ is a response-field index from $u$ and $j$ is a physical-field index from $v$.

In [10]:
# External fields for the 2-point function <dn1 dn2>
# dn1, dn2 are physical fields — they sit at the incoming-edge end of leaves
external_fields = [('dn', 1), ('dn', 2)]

# Build field index maps
ring_var_names_list = list(ns._ring_var_names)
n_tilde = ft._n_tilde
resp_idx, phys_idx = build_field_index_map(ring_var_names_list, n_tilde)

print('Response field index map:')
for leg, idx in sorted(resp_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} → row {idx}')
print()
print('Physical field index map:')
for leg, idx in sorted(phys_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} → col {idx}')

# Helper to display typed diagrams nicely
def show_typed_diagram(td, idx):
    D = td.prediagram[0]
    leaves = td.prediagram[2]
    print(f'\n{"="*60}')
    print(f'Diagram {idx}')
    print(f'{"="*60}')
    
    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} ← {field[0]}{field[1]}')
    
    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            print(f'    v{v} ({tname}): bigrade={vtype.bigrade}, '
                  f'resp={vtype.response_legs}', end='')
            if hasattr(vtype, 'physical_legs'):
                print(f', phys={vtype.physical_legs}', end='')
            print()
            display(Math(f'\\quad\\text{{coeff}} = {latex(vtype.coefficient)}'))
    
    print('  Edges (propagator assignments):')
    for (u, v), (resp_leg, phys_leg) in sorted(td.edge_types.items()):
        ri, pi = td.propagator_indices.get((u, v), ('?', '?'))
        print(f'    {u} → {v}:  {resp_leg[0]}{resp_leg[1]} → {phys_leg[0]}{phys_leg[1]}  '
              f'[G_hat[{ri},{pi}]]')

Response field index map:
  ('nt', 1) → row 0
  ('nt', 2) → row 1
  ('vt', 1) → row 2
  ('vt', 2) → row 3

Physical field index map:
  ('dn', 1) → col 0
  ('dn', 2) → col 1
  ('dv', 1) → col 2
  ('dv', 2) → col 3


### 5.1 Tree-level typed diagrams

In [ ]:
def _type_tree():
    return enumerate_all_typed(
        kept_0, external_fields, vtypes, stypes, G_ft, resp_idx, phys_idx
    )

typed_tree = cache.get_or_compute(
    'typed', _type_tree, k=k, loop_order=0
) if USE_CACHE else _type_tree()

print(f'Tree-level: {len(kept_0)} prediagrams → {len(typed_tree)} typed diagrams')

for i, td in enumerate(typed_tree, 1):
    show_typed_diagram(td, f'Tree-{i}')

### 5.2 One-loop typed diagrams

In [ ]:
def _type_1loop():
    return enumerate_all_typed(
        kept_1, external_fields, vtypes, stypes, G_ft, resp_idx, phys_idx
    )

typed_1loop = cache.get_or_compute(
    'typed', _type_1loop, k=k, loop_order=1
) if USE_CACHE else _type_1loop()

print(f'1-loop: {len(kept_1)} prediagrams → {len(typed_1loop)} typed diagrams')

for i, td in enumerate(typed_1loop, 1):
    show_typed_diagram(td, f'1L-{i}')

---
## 6. Unique Diagrams & Combinatorial Factor $M(\Gamma)$

Deduplicate typed diagrams to obtain the set of unique Feynman diagrams $\Gamma$. For each, compute the **combinatorial factor**

$$M(\Gamma) = \prod_{v} \prod_{\substack{\text{groups of } k \\ \text{identical legs at } v}} k!$$

which counts the number of distinct valid **attachments** — bijections from each vertex's leg multiset to its incident edges — that realize $\Gamma$.

The diagram's contribution to the $k$-point function is:

$$\text{weight}(\Gamma) = M(\Gamma) \times \prod_{v} \text{coeff}(v) \times \int(\text{propagators})$$

The vertex coefficients already contain $1/n!$ from the Taylor expansion, and prediagram isomorphism removal already accounts for the $1/p!$ from the perturbative expansion. $M(\Gamma)$ is the remaining combinatorial counting factor (cf. Helias & Dahmen, Ch. 9).

In [ ]:
from msrjd.diagrams.symmetry import classify_coefficient_factors

# Get time-dependence metadata from the model
time_dep_params = HAWKES_MODEL.get('time_dependent_parameters', [])
noise_structure = HAWKES_MODEL.get('noise_structure', {
    'temporal_type': 'white', 'amplitude_params': []
})
print(f'Noise temporal type: {noise_structure.get("temporal_type", "white")}')
print(f'Noise amplitude params: {noise_structure.get("amplitude_params", [])}')
print(f'Time-dependent params: {time_dep_params if time_dep_params else "(none -- stationary)"}')
print()


def show_unique_diagram(td, idx, winfo):
    """Display a unique diagram with M(Gamma) and weight structure."""
    M = winfo['M']
    print(f'\n{"="*60}')
    print(f'Unique Diagram {idx}    M = {M}')
    print(f'{"="*60}')

    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} <- {field[0]}{field[1]}')

    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            print(f'    v{v} ({tname}): bigrade={vtype.bigrade}, '
                  f'resp={vtype.response_legs}', end='')
            if hasattr(vtype, 'physical_legs'):
                print(f', phys={vtype.physical_legs}', end='')
            print()
            display(Math(f'\\quad c_{{v_{v}}} = {latex(vtype.coefficient)}'))

    print('  Edges (propagator assignments):')
    for edge_key in sorted(td.edge_types.keys()):
        resp_leg, phys_leg = td.edge_types[edge_key]
        u, v = edge_key[0], edge_key[1]
        ri, pi = td.propagator_indices.get(edge_key, ('?', '?'))
        print(f'    {u} -> {v}:  {resp_leg[0]}{resp_leg[1]} -> '
              f'{phys_leg[0]}{phys_leg[1]}  [G_hat[{ri},{pi}]]')

    sp = winfo['scalar_prefactor']
    display(Math(
        r'\text{Scalar prefactor: }\;' + latex(sp)
    ))


# ── Tree level ──
print('='*60)
print('TREE-LEVEL UNIQUE DIAGRAMS')
print('='*60)

unique_tree = cache.get_or_compute(
    'unique_typed', lambda: deduplicate_typed_diagrams(typed_tree), k=k, loop_order=0
) if USE_CACHE else deduplicate_typed_diagrams(typed_tree)

print(f'{len(typed_tree)} typed diagrams -> {len(unique_tree)} unique diagrams')

for i, td in enumerate(unique_tree, 1):
    winfo = classify_coefficient_factors(td, time_dep_params, noise_structure)
    show_unique_diagram(td, f'Tree-{i}', winfo)


# ── 1-loop ──
print('\n')
print('='*60)
print('1-LOOP UNIQUE DIAGRAMS')
print('='*60)

unique_1loop = cache.get_or_compute(
    'unique_typed', lambda: deduplicate_typed_diagrams(typed_1loop), k=k, loop_order=1
) if USE_CACHE else deduplicate_typed_diagrams(typed_1loop)

print(f'{len(typed_1loop)} typed diagrams -> {len(unique_1loop)} unique diagrams')

for i, td in enumerate(unique_1loop, 1):
    winfo = classify_coefficient_factors(td, time_dep_params, noise_structure)
    show_unique_diagram(td, f'1L-{i}', winfo)

---
## 7. Symbolic Integration (Stationary Case)

For each unique diagram $\Gamma$, construct the **unevaluated frequency-domain
integral** that gives the time-domain contribution $C_\Gamma(t_1, t_2)$.

### Procedure

1. **Assign** a frequency variable $\omega_{e_i}$ to each directed edge.

2. **Solve conservation** at every internal vertex (interaction and source alike):
   $$\delta\!\Big(\sum_{\text{in}} \omega_e - \sum_{\text{out}} \omega_e\Big)$$
   This expresses internal edge frequencies in terms of external ones.
   If the diagram has $\ell$ loops, $\ell$ internal frequencies remain free.

3. **Build the integrand**: for each edge, substitute the resolved frequency
   into the propagator entry $\hat{G}_{i,j}(\omega_e)$.  Each external leg
   contributes an exponential $e^{\pm i\omega t}$ from the inverse Fourier
   transform (sign from leaf directionality: tail $\to e^{-i\omega t}$,
   head $\to e^{+i\omega t}$).

4. **Display** the full unevaluated integral:
   $$C_\Gamma(t_1, t_2) = \text{scalar\_pf}
   \;\prod_j \int\!\frac{d\omega_j}{2\pi}\;
   \Bigl[\prod_e \hat{G}_{i_e,j_e}(\omega_e)\Bigr]
   \Bigl[\prod_{\text{ext}} e^{\pm i\omega t}\Bigr]$$

In [ ]:
from msrjd.integration.symbolic import (
    check_propagator_available,
    build_integrand_stationary,
    extract_propagator_factors,
    canonicalize_prop_factors,
    loop_kernel_signature,
    group_diagrams_by_kernel,
)

# ── Propagator data dict (assembled from Section 1.2) ──
omega = SR.var('omega', latex_name=r'\omega')

propagator_data = {
    'G_ft': G_ft,
    'adj_ft': adj_ft,
    'D_omega': D_omega,
    'G_ft_explicit': True,
    'pole_vals': pole_vals,
    'C_mats': C_mats,
    'nf': nf,
}

prop_mode = check_propagator_available(propagator_data)
print(f'Propagator mode: {prop_mode}')
print(f'Poles: {pole_vals}')
print()


def show_integral(td, label, propagator_data, omega_sym):
    """
    Build and display the unevaluated integral for a typed diagram.
    Returns the integrand result dict.
    """
    ir = build_integrand_stationary(
        td, propagator_data, k=2,
        omega_symbol=omega_sym,
        time_dep_params=HAWKES_MODEL.get('time_dependent_parameters', []),
        noise_structure=HAWKES_MODEL.get('noise_structure'),
    )

    prefactor = ir['scalar_prefactor']
    full_integrand = ir['full_integrand']
    ext_freqs = ir['ext_freqs']
    free_freqs = ir['free_freqs']
    ell = ir['loop_number']
    overall = ir['overall_conservation']

    print(f'\n{"="*60}')
    print(f'{label}   (ell = {ell})')
    print(f'{"="*60}')

    # Frequency assignments
    print(f'  Free (loop) frequencies: {free_freqs if free_freqs else "(none)"}')
    print(f'  Independent external frequencies: {ext_freqs}')

    if overall is not None:
        display(Math(
            r'  \text{Overall conservation: }\;'
            + latex(overall) + r' = 0'
        ))

    # ── Unevaluated integral ──
    int_vars = list(free_freqs) + list(ext_freqs)
    integrals_tex = ''
    for v in int_vars:
        integrals_tex += r'\int\!\frac{d' + latex(v) + r'}{2\pi}\;'

    pf_tex = latex(prefactor)
    integrand_tex = latex(full_integrand)

    display(Math(
        r'C_{\Gamma}(t_1, t_2) = '
        + pf_tex + r'\;'
        + integrals_tex
        + integrand_tex
    ))

    # Factored view
    integrand_only = ir['integrand']
    display(Math(
        r'\text{Propagator product: }\;'
        + latex(integrand_only)
    ))

    exp_factor = (full_integrand / integrand_only)
    try:
        exp_factor = exp_factor.simplify()
    except Exception:
        pass
    display(Math(
        r'\text{Exponential factor: }\;'
        + latex(exp_factor)
    ))

    return ir


# ═══════════════════════════════════════════════════════════════
# 7.1  All tree-level diagrams
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('TREE-LEVEL INTEGRALS')
print('='*60)

ir_trees = []
for i, td in enumerate(unique_tree, 1):
    ir_i = show_integral(td, f'Tree-{i}', propagator_data, omega)
    ir_trees.append(ir_i)


# ═══════════════════════════════════════════════════════════════
# 7.2  All 1-loop diagrams
# ═══════════════════════════════════════════════════════════════
print('\n')
print('='*60)
print('1-LOOP INTEGRALS')
print('='*60)

ir_loops = []
for i, td in enumerate(unique_1loop, 1):
    ir_i = show_integral(td, f'1-Loop-{i}', propagator_data, omega)
    ir_loops.append(ir_i)

### 7.2 Unique loop kernels

Group all diagrams by their **loop kernel** -- the product of propagator entries
$\prod_e \hat{G}_{i_e,j_e}(\omega_e)$ with the same frequency routing.

Diagrams sharing a loop kernel differ only in their scalar prefactors (vertex
coefficients, combinatorial factors).  These prefactors simply add, so we only
need to numerically integrate each unique kernel once.

For each unique kernel, we show:
- Which diagrams it came from
- The combined scalar prefactor
- The propagator factors (opaque representation)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Group ALL diagrams (tree + 1-loop) by loop kernel
# ═══════════════════════════════════════════════════════════════
all_unique = list(unique_tree) + list(unique_1loop)

kernel_groups = group_diagrams_by_kernel(
    all_unique, propagator_data, k=2,
    omega_symbol=omega,
    time_dep_params=HAWKES_MODEL.get('time_dependent_parameters', []),
    noise_structure=HAWKES_MODEL.get('noise_structure'),
)

print(f'Total unique diagrams: {len(all_unique)}')
print(f'Unique loop kernels:   {len(kernel_groups)}')
print(f'Numerical integrations saved: {len(all_unique) - len(kernel_groups)}')
print()

for g_idx, g in enumerate(kernel_groups, 1):
    ell = g['loop_number']
    n = g['n_diagrams']
    combined_pf = g['combined_prefactor']

    print(f'{"="*60}')
    print(f'Kernel {g_idx}   (ell = {ell}, {n} diagram{"s" if n > 1 else ""})')
    print(f'{"="*60}')

    # Show which diagrams contribute
    for j, td in enumerate(g['diagrams']):
        pf_j = g['individual_prefactors'][j]
        # Identify tree vs loop
        is_tree = td in unique_tree
        tree_idx = unique_tree.index(td) + 1 if is_tree else None
        loop_idx = unique_1loop.index(td) + 1 if not is_tree else None
        label = f'Tree-{tree_idx}' if is_tree else f'1-Loop-{loop_idx}'
        print(f'  {label}:  prefactor = {pf_j}')

    # Combined prefactor
    display(Math(
        r'\text{Combined prefactor: }\;'
        + latex(combined_pf)
    ))

    # Show propagator factors in opaque form
    pf_list = g['prop_factors']
    factor_strs = []
    for f in pf_list:
        if f[0] == 'prop':
            _, ri, pi, omega_expr = f
            factor_strs.append(
                r'\hat{G}_{' + str(ri) + ',' + str(pi) + r'}('
                + latex(omega_expr) + ')'
            )
        elif f[0] == 'noise':
            factor_strs.append(
                r'\hat{\kappa}(' + latex(f[1]) + ')'
            )

    if factor_strs:
        product_tex = r' \cdot '.join(factor_strs)
        display(Math(
            r'\text{Propagator product: }\;' + product_tex
        ))

    # Show the integrand from the representative
    ir = g['representative_ir']
    if ell > 0:
        loop_vars_tex = ', '.join(latex(v) for v in ir['free_freqs'])
        ext_vars_tex = ', '.join(latex(v) for v in ir['ext_freqs'])
        print(f'  Loop variables: {loop_vars_tex}')
        print(f'  External variables: {ext_vars_tex}')

    print()

### 7.3 Numerical evaluation -- spectral grid + inverse FFT

For each unique loop kernel, numerically evaluate the frequency-domain
spectrum $\hat{C}(\omega)$ on a grid, then inverse Fourier transform
to get $C(\tau)$.

**Strategy** (using kernel deduplication):

1. For each unique kernel group, evaluate the loop integral once.
2. Multiply by the combined scalar prefactor.
3. Sum all kernel contributions to get the total spectrum.
4. Inverse FFT to get $C(\tau)$.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.3  Numerical evaluation: spectral grid + inverse FFT
# ═══════════════════════════════════════════════════════════════
import numpy as np
from sage.all import numerical_integral, CC, fast_callable, CDF

# ── Concrete parameters (quadratic phi, 2-population Hawkes) ──
num_params = {
    SR.var('nstar1'): 5.0,
    SR.var('nstar2'): 3.0,
    SR.var('w11'): 0.1,
    SR.var('w12'): 0.05,
    SR.var('w21'): 0.08,
    SR.var('w22'): 0.06,
    SR.var('tau'): 10.0,
    SR.var('phi1_1'): 0.5,
    SR.var('phi1_2'): 0.4,
    SR.var('phi2_1'): 0.02,
    SR.var('phi2_2'): 0.015,
}

print('Numerical parameters:')
for sym, val in num_params.items():
    print(f'  {sym} = {val}')

# Check pole stability
print(f'\nPoles (numerical):')
for p_idx, p in enumerate(pole_vals):
    p_num = complex(CC(p.subs(num_params)))
    print(f'  pole {p_idx+1}: {p_num:.6f}   (Im = {p_num.imag:.4f})')


# ── Grid design ──
T_max = 80.0
Delta_tau = 0.25
N_fft = int(2 * T_max / Delta_tau)
N_fft = int(2**np.ceil(np.log2(N_fft)))
Delta_omega = 2 * np.pi / (N_fft * Delta_tau)
omega_max = N_fft * Delta_omega / 2

omega_grid = np.arange(-N_fft//2, N_fft//2) * Delta_omega

print(f'\nGrid: N={N_fft}, omega_max={omega_max:.2f}, '
      f'Delta_omega={Delta_omega:.4f}, Delta_tau={Delta_tau}')
print(f'Tau range: [{-T_max:.1f}, {T_max:.1f}]')


# ── Evaluation helpers ──

def _find_integrand_vars(ir, num_params):
    """
    Substitute num_params into ir['integrand'] and identify which
    omega variables remain. Returns (integrand_num, ext_var, free_vars).
    """
    integrand_num = ir['integrand'].subs(num_params)
    remaining = sorted(integrand_num.variables(), key=str)

    ext_set = set(ir['ext_freqs'])
    free_set = set(ir['free_freqs'])

    omega_ext_in = [v for v in remaining if v in ext_set]
    omega_free_in = [v for v in remaining if v in free_set]
    other = [v for v in remaining if v not in ext_set and v not in free_set]

    if omega_ext_in:
        ext_var = omega_ext_in[0]
        loop_vars = omega_free_in + other
    elif len(remaining) >= 2:
        non_free = [v for v in remaining if v not in free_set]
        if non_free:
            ext_var = non_free[0]
            loop_vars = [v for v in remaining if v != ext_var]
        else:
            ext_var = remaining[0]
            loop_vars = remaining[1:]
    elif len(remaining) == 1:
        ext_var = remaining[0]
        loop_vars = []
    else:
        ext_var = None
        loop_vars = []

    return integrand_num, ext_var, loop_vars


def spectrum_tree(ir, num_params, omega_grid):
    """Evaluate tree-level spectrum Chat(omega) on grid."""
    prefactor = float(ir['scalar_prefactor'].subs(num_params))
    integrand_num, ext_var, loop_vars = _find_integrand_vars(ir, num_params)

    if ext_var is None:
        print('  Tree: no omega variable found!')
        return np.zeros(len(omega_grid), dtype=complex)

    f_fast = fast_callable(integrand_num, vars=[ext_var], domain=CDF)
    Chat = np.array([complex(f_fast(w)) for w in omega_grid])
    return prefactor * Chat


def spectrum_one_loop(ir, num_params, omega_grid, Omega_max=50, n_quad=1000):
    """
    Evaluate one-loop spectrum Chat(omega) on grid.
    For each external omega, numerically integrate the loop frequency.
    """
    prefactor = float(ir['scalar_prefactor'].subs(num_params))
    integrand_num, ext_var, loop_vars = _find_integrand_vars(ir, num_params)

    if not loop_vars:
        print('  1-loop: no loop variable found -- treating as tree')
        return spectrum_tree(ir, num_params, omega_grid)

    omega_loop_var = loop_vars[0]

    if ext_var is not None:
        f_fast = fast_callable(integrand_num, vars=[ext_var, omega_loop_var], domain=CDF)
    else:
        f_fast = fast_callable(integrand_num, vars=[omega_loop_var], domain=CDF)

    Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
    dOmega = Omega_pts[1] - Omega_pts[0]

    Chat = np.zeros(len(omega_grid), dtype=complex)
    for i, w_ext in enumerate(omega_grid):
        if ext_var is not None:
            vals = np.array([complex(f_fast(w_ext, Om)) for Om in Omega_pts])
        else:
            vals = np.array([complex(f_fast(Om)) for Om in Omega_pts])
        Chat[i] = np.trapz(vals, dx=dOmega) / (2 * np.pi)

    return prefactor * Chat


def inverse_fourier(Chat, omega_grid):
    """Inverse DFT: C(tau) from Chat(omega)."""
    N = len(omega_grid)
    Delta_omega = omega_grid[1] - omega_grid[0]
    Chat_shifted = np.fft.ifftshift(Chat)
    C_tau = np.fft.fftshift(np.fft.ifft(Chat_shifted)) * N * Delta_omega / (2 * np.pi)
    tau_grid = np.fft.fftshift(np.fft.fftfreq(N, d=Delta_omega / (2 * np.pi)))
    return tau_grid, C_tau


def evaluate_kernel_group(g, num_params, omega_grid, Omega_max=50, n_quad=1000):
    """
    Evaluate a kernel group: compute the loop integral once,
    multiply by the combined prefactor.
    """
    ir = g['representative_ir']
    ell = g['loop_number']
    combined_pf = g['combined_prefactor']

    # Replace the representative's prefactor with the combined one
    ir_eval = dict(ir)
    ir_eval['scalar_prefactor'] = combined_pf

    if ell == 0:
        return spectrum_tree(ir_eval, num_params, omega_grid)
    else:
        return spectrum_one_loop(ir_eval, num_params, omega_grid,
                                 Omega_max=Omega_max, n_quad=n_quad)


# ═══════════════════════════════════════════════════════════════
# Evaluate all kernel groups
# ═══════════════════════════════════════════════════════════════
print(f'\n{"="*60}')
print(f'Evaluating {len(kernel_groups)} unique kernel(s)')
print(f'{"="*60}')

Chat_total = np.zeros(N_fft, dtype=complex)
Chat_tree_total = np.zeros(N_fft, dtype=complex)
Chat_loop_total = np.zeros(N_fft, dtype=complex)
mid = N_fft // 2

for g_idx, g in enumerate(kernel_groups, 1):
    ell = g['loop_number']
    n = g['n_diagrams']
    print(f'\nKernel {g_idx}/{len(kernel_groups)}  '
          f'(ell={ell}, {n} diagram{"s" if n > 1 else ""})')

    Chat_g = evaluate_kernel_group(g, num_params, omega_grid)
    Chat_total += Chat_g

    if ell == 0:
        Chat_tree_total += Chat_g
    else:
        Chat_loop_total += Chat_g

    print(f'  Chat(0) = {Chat_g[mid]:.6e}')
    print(f'  max|Chat| = {np.nanmax(np.abs(Chat_g)):.6e}')

# Summary
print(f'\n{"="*60}')
print('Summary')
print(f'{"="*60}')
print(f'  Tree Chat(0)  = {Chat_tree_total[mid]:.6e}')
print(f'  Loop Chat(0)  = {Chat_loop_total[mid]:.6e}')
print(f'  Total Chat(0) = {Chat_total[mid]:.6e}')


# ═══════════════════════════════════════════════════════════════
# Inverse FFT
# ═══════════════════════════════════════════════════════════════
print(f'\n{"="*60}')
print('Inverse Fourier transform')
print(f'{"="*60}')

tau_out, C_tree = inverse_fourier(Chat_tree_total, omega_grid)
print(f'  Tree C(0) = {C_tree[mid].real:.6e}')

C_loop_tau = None
if not np.all(Chat_loop_total == 0):
    _, C_loop_tau = inverse_fourier(Chat_loop_total, omega_grid)
    print(f'  Loop C(0) = {C_loop_tau[mid].real:.6e}')

_, C_total = inverse_fourier(Chat_total, omega_grid)
print(f'  Total C(0) = {C_total[mid].real:.6e}')


# ═══════════════════════════════════════════════════════════════
# Plot
# ═══════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: frequency-domain spectrum
ax = axes[0]
ax.plot(omega_grid, Chat_tree_total.real, 'b-', lw=1.5,
        label=r'Tree Re $\hat{C}$')
if not np.all(Chat_loop_total == 0):
    ax.plot(omega_grid, Chat_loop_total.real, 'r-', lw=1.5,
            label=r'Loop Re $\hat{C}$')
ax.set_xlabel(r'$\omega$', fontsize=13)
ax.set_ylabel(r'$\hat{C}(\omega)$', fontsize=13)
ax.set_title('Frequency-domain spectrum', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 2)

# Right: C(tau)
ax = axes[1]
mask = np.abs(tau_out) <= 50
ax.plot(tau_out[mask], C_tree[mask].real, 'b-', lw=2,
        label=r'Tree $C(\tau)$')
if C_loop_tau is not None:
    ax.plot(tau_out[mask], C_total[mask].real, 'k-', lw=2,
            label=r'Tree + Loop $C(\tau)$')
    ax.plot(tau_out[mask], C_loop_tau[mask].real, 'r-', lw=1.5,
            label=r'Loop correction')
ax.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
ax.set_ylabel(r'$C(\tau)$', fontsize=13)
ax.set_title(r'Covariance $C(\tau)$', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()